# DiffDock 虚拟筛选教程

**论文**: Corso, G., Staerk, H., Jing, B., Barzilay, R., & Jaakkola, T. (2023). *DiffDock: Diffusion Steps, Twists, and Turns for Molecular Docking.* ICLR 2023.

---

## 核心创新

DiffDock 是首个将 **扩散生成模型 (Diffusion Generative Model)** 应用于分子对接的方法。其核心创新在于：

- 将配体在蛋白口袋中的位姿 (pose) 分解为 **三个独立的自由度通道**：
  1. **平移 (Translation, R3)**：配体质心在三维空间中的位移
  2. **旋转 (Rotation, SO(3))**：配体绕质心的刚体旋转
  3. **扭转 (Torsion, S1)**：配体内部可旋转键的二面角变化

- 对每个通道分别施加 **前向扩散（加噪）** 和 **逆向去噪** 过程
- 训练 **去噪分数模型 (Denoising Score Model)** 学习每个通道的去噪方向
- 推理时从随机位姿出发，通过迭代去噪恢复正确的结合构象

这种 SE(3) 空间上的三通道扩散设计，使得模型能够同时处理刚体变换（平移+旋转）和分子柔性（扭转），是对传统对接方法的根本性创新。

## 学习目标

通过本教程，你将学会：

1. 理解 SE(3) 扩散模型的三通道噪声调度机制
2. 实现前向扩散（对平移、旋转、扭转分别加噪）
3. 构建去噪分数预测网络（蛋白-配体交互 + 三头预测）
4. 使用去噪分数匹配 (Denoising Score Matching) 训练模型
5. 实现逆扩散推理（从随机位姿迭代恢复正确构象）
6. 评估对接结果（RMSD 分布、去噪轨迹可视化）

In [ ]:
# ============================
# 环境设置、路径配置与依赖导入
# ============================

import os
import math
import warnings
from pathlib import Path
from collections import deque

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from rdkit import Chem
from rdkit import RDLogger

%matplotlib inline
import matplotlib.pyplot as plt

# 抑制 RDKit 的冗余警告信息
RDLogger.logger().setLevel(RDLogger.ERROR)
warnings.filterwarnings('ignore')


def find_project_root(marker='demo_data'):
    """向上查找包含 demo_data/ 的项目根目录"""
    here = Path('.').resolve()
    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f'无法找到包含 {marker}/ 的项目根目录')


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / 'demo_data'
CORESET_FILE = DATA_DIR / 'CoreSet.dat'
COMPLEX_DIR = DATA_DIR / 'coreset'

print(f"项目根目录: {PROJECT_ROOT}")
print(f"数据目录:   {DATA_DIR}")

# 版本信息
version_info = pd.DataFrame({
    '库': ['torch', 'numpy', 'rdkit', 'pandas'],
    '版本': [
        torch.__version__,
        np.__version__,
        Chem.rdBase.rdkitVersion,
        pd.__version__,
    ]
})
display(version_info)

## 1. 超参数设置

DiffDock 的超参数分为两大类：

### 模型与训练参数
- `DISTANCE_CUTOFF`：蛋白-配体交互距离阈值，决定哪些原子对之间构建消息传递边
- `HIDDEN_DIM`：隐层维度，控制模型容量
- `N_EPOCHS`：训练轮数
- `LR`：学习率，扩散模型通常使用较小的学习率以保证训练稳定性

### 噪声调度参数（DiffDock 特有）
三个通道使用不同的 sigma_min 和 sigma_max，因为平移、旋转、扭转的物理尺度和分布特性不同：
- **平移**: 以 Angstrom 为单位，范围较大 -> sigma in [0.1, 10.0]
- **旋转**: 以弧度为单位，范围适中 -> sigma in [0.1, 1.5]
- **扭转**: 以弧度为单位，范围 [-pi, pi] -> sigma in [0.01, pi]

噪声调度公式（指数调度）：

    sigma(t) = sigma_min^(1-t) * sigma_max^t,  t in [0, 1]

当 t=0 时 sigma = sigma_min（几乎无噪声），当 t=1 时 sigma = sigma_max（完全噪声）。

In [ ]:
# ============================
# 超参数定义
# ============================

# ---------- 模型与训练超参数 ----------
DISTANCE_CUTOFF = 5.0       # 蛋白-配体交互距离阈值 (Angstrom)
ATOM_FEAT_DIM = 10          # 简化原子特征维度
HIDDEN_DIM = 128            # 隐层维度
N_EPOCHS = 200              # 训练轮数
LR = 1e-4                   # 学习率 (扩散模型用较小学习率，更稳定)
BATCH_SIZE = 1              # 变长图，逐样本处理
SEED = 42
N_STEPS = 20                # 逆扩散推理步数

# ---------- 噪声调度参数 ----------
# DiffDock 对平移、旋转、扭转使用独立的噪声调度
TR_SIGMA_MIN = 0.1          # 平移噪声最小标准差
TR_SIGMA_MAX = 10.0         # 平移噪声最大标准差
ROT_SIGMA_MIN = 0.1         # 旋转噪声最小标准差
ROT_SIGMA_MAX = 1.5         # 旋转噪声最大标准差
TOR_SIGMA_MIN = 0.01        # 扭转噪声最小标准差
TOR_SIGMA_MAX = np.pi       # 扭转噪声最大标准差

# ---------- 设备选择 ----------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"计算设备: {DEVICE}")
print(f"扩散模型噪声调度:")
print(f"  平移 sigma: [{TR_SIGMA_MIN}, {TR_SIGMA_MAX}]")
print(f"  旋转 sigma: [{ROT_SIGMA_MIN}, {ROT_SIGMA_MAX}]")
print(f"  扭转 sigma: [{TOR_SIGMA_MIN}, {TOR_SIGMA_MAX:.4f}]")

## 2. 数据加载与特征提取

本节包含以下内容：

### 2.1 数据解析
从 PDBbind CASF-2016 核心集中解析蛋白-配体复合物标签（结合亲和力 logKa）。

### 2.2 原子特征提取
构建 10 维原子特征向量：
- 前 9 维：元素类型 one-hot 编码（C, N, O, S, F, P, Cl, Br, other）
- 第 10 维：是否为芳香性原子

### 2.3 扩散工具函数
这是 DiffDock 的核心创新所在，包括：

- **`SinusoidalEmbedding`**：时间步的正弦位置编码，与 Transformer 的位置编码原理相同
- **`t_to_sigma`**：指数噪声调度，将归一化时间步映射为三通道噪声标准差
- **`random_rotation_matrix`**：通过 QR 分解生成均匀分布的随机旋转矩阵
- **`axis_angle_to_matrix`**：Rodrigues 公式，将轴角向量转换为旋转矩阵
- **`get_rotatable_bonds`**：获取配体的可旋转键列表
- **`modify_torsion_angles`**：通过 BFS + Rodrigues 公式修改配体扭转角
- **`add_noise`**：三通道独立加噪（平移 + 旋转 + 扭转）

In [ ]:
# ============================
# 数据解析与特征提取函数
# ============================

# ---- 元素符号 -> one-hot 映射 (9类 + 1 芳香性 = 10维) ----
ELEMENT_LIST = ['C', 'N', 'O', 'S', 'F', 'P', 'Cl', 'Br']  # 8 种常见元素 + 1 other


def parse_coreset(path):
    """解析 CoreSet.dat，返回 {pdbid: logKa} 字典。跳过以 '#' 开头的注释行。"""
    labels = {}
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split()
            pdbid = parts[0]
            logKa = float(parts[3])
            labels[pdbid] = logKa
    print(f"从 CoreSet.dat 读取到 {len(labels)} 个复合物标签")
    return labels


def atom_features(atom):
    """
    构建 10 维原子特征向量：
      - 前 9 维: 元素类型 one-hot (C, N, O, S, F, P, Cl, Br, other)
      - 第 10 维: 是否为芳香性原子
    """
    feat = np.zeros(ATOM_FEAT_DIM, dtype=np.float32)
    symbol = atom.GetSymbol()
    if symbol in ELEMENT_LIST:
        feat[ELEMENT_LIST.index(symbol)] = 1.0
    else:
        feat[8] = 1.0       # other 类别
    feat[9] = float(atom.GetIsAromatic())
    return feat


def load_mol(path, fmt):
    """
    用 RDKit 加载分子文件。
    fmt: 'mol2', 'sdf', 'pdb'
    先尝试正常加载，再尝试 sanitize=False 后手动 sanitize。
    """
    mol = None
    if fmt == 'mol2':
        mol = Chem.MolFromMol2File(path, sanitize=True)
        if mol is None:
            mol = Chem.MolFromMol2File(path, sanitize=False)
            if mol is not None:
                try:
                    Chem.SanitizeMol(mol)
                except Exception:
                    pass
    elif fmt == 'sdf':
        supplier = Chem.SDMolSupplier(path, sanitize=True)
        for m in supplier:
            if m is not None:
                mol = m
                break
        if mol is None:
            supplier = Chem.SDMolSupplier(path, sanitize=False)
            for m in supplier:
                if m is not None:
                    mol = m
                    try:
                        Chem.SanitizeMol(mol)
                    except Exception:
                        pass
                    break
    elif fmt == 'pdb':
        mol = Chem.MolFromPDBFile(path, removeHs=False, sanitize=True)
        if mol is None:
            mol = Chem.MolFromPDBFile(path, removeHs=False, sanitize=False)
            if mol is not None:
                try:
                    Chem.SanitizeMol(mol)
                except Exception:
                    pass
    return mol

In [ ]:
# ============================
# 扩散工具函数
# ============================
# DiffDock 的核心创新: 将 SE(3) 空间的扩散拆分为三个独立通道


class SinusoidalEmbedding(nn.Module):
    """
    时间步的正弦位置编码 -- 将标量时间步映射为高维向量

    与 Transformer 中的位置编码原理相同：
    用不同频率的正弦/余弦函数将标量映射到高维空间，
    让模型能够区分不同的时间步（噪声水平）。
    """
    def __init__(self, dim=HIDDEN_DIM):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        """t: 标量或 (B,) 张量 -> (B, dim) 嵌入向量"""
        if not isinstance(t, torch.Tensor):
            t = torch.tensor([t], dtype=torch.float32)
        if t.dim() == 0:
            t = t.unsqueeze(0)
        device = t.device
        half_dim = self.dim // 2
        emb = math.log(10000.0) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, dtype=torch.float32, device=device) * -emb)
        emb = t.float().unsqueeze(-1) * emb.unsqueeze(0)   # (B, half_dim)
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)  # (B, dim)
        return emb


def t_to_sigma(t):
    """
    指数噪声调度: 将归一化时间步 t in [0,1] 映射为三通道噪声标准差

    公式: sigma(t) = sigma_min^(1-t) * sigma_max^t

    当 t=0 时 sigma=sigma_min（几乎无噪声），当 t=1 时 sigma=sigma_max（完全噪声）。
    三个通道使用不同的 sigma_min/sigma_max，因为平移、旋转、扭转
    的物理尺度和分布特性不同：
      - 平移: 以 Angstrom 为单位，范围较大
      - 旋转: 以弧度为单位，范围适中
      - 扭转: 以弧度为单位，范围 [-pi, pi]
    """
    tr_sigma = TR_SIGMA_MIN ** (1 - t) * TR_SIGMA_MAX ** t
    rot_sigma = ROT_SIGMA_MIN ** (1 - t) * ROT_SIGMA_MAX ** t
    tor_sigma = TOR_SIGMA_MIN ** (1 - t) * TOR_SIGMA_MAX ** t
    return tr_sigma, rot_sigma, tor_sigma


def random_rotation_matrix():
    """
    通过 QR 分解生成均匀分布的随机旋转矩阵

    方法: 对随机高斯矩阵做 QR 分解，Q 矩阵即为均匀分布的正交矩阵。
    需要修正符号以确保 det(Q) = +1（旋转而非反射）。
    """
    M = np.random.randn(3, 3)
    Q, R = np.linalg.qr(M)
    Q = Q @ np.diag(np.sign(np.diag(R)))
    if np.linalg.det(Q) < 0:
        Q[:, 0] *= -1
    return Q.astype(np.float32)


def axis_angle_to_matrix(axis_angle):
    """
    Rodrigues 公式: 轴角向量 -> 旋转矩阵

    输入: axis_angle (3,) numpy 数组，方向为旋转轴，模长为旋转角度（弧度）
    输出: (3, 3) 旋转矩阵

    公式: R = I + sin(theta)*K + (1-cos(theta))*K^2
    其中 K 是旋转轴 k 的反对称矩阵:
        K = [[  0, -k3,  k2],
             [ k3,   0, -k1],
             [-k2,  k1,   0]]
    """
    axis_angle = np.asarray(axis_angle, dtype=np.float64)
    angle = np.linalg.norm(axis_angle)
    if angle < 1e-8:
        return np.eye(3, dtype=np.float32)
    k = axis_angle / angle                  # 单位旋转轴
    K = np.array([[0, -k[2], k[1]],         # 反对称矩阵
                  [k[2], 0, -k[0]],
                  [-k[1], k[0], 0]])
    R = np.eye(3) + np.sin(angle) * K + (1 - np.cos(angle)) * (K @ K)
    return R.astype(np.float32)


def get_rotatable_bonds(mol):
    """
    获取配体的可旋转键列表（非环单键，不含氢原子）

    可旋转键定义:
      - 单键（非双键、三键、芳香键）
      - 不在环中（环内键旋转会破坏环结构）
      - 两端原子都不是氢（氢原子的旋转无物理意义）
      - 两端原子的度数 > 1（末端原子旋转无意义）
    """
    rot_bonds = []
    for bond in mol.GetBonds():
        if (bond.GetBondType() == Chem.rdchem.BondType.SINGLE
                and not bond.IsInRing()
                and bond.GetBeginAtom().GetAtomicNum() != 1
                and bond.GetEndAtom().GetAtomicNum() != 1
                and bond.GetBeginAtom().GetDegree() > 1
                and bond.GetEndAtom().GetDegree() > 1):
            rot_bonds.append((bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()))
    return rot_bonds


def modify_torsion_angles(coords, mol, rot_bonds, delta_angles):
    """
    修改配体扭转角: 对每个可旋转键，用 BFS 确定旋转侧，再用 Rodrigues 公式旋转

    算法:
      对于每个可旋转键 (u, v) 和对应的旋转角度 delta_theta:
        1. 从 v 出发做 BFS（不经过 u），找到 v 侧的所有原子
        2. 旋转轴方向 = u -> v
        3. 用 Rodrigues 公式将 v 侧原子绕旋转轴旋转 delta_theta（以 u 为支点）

    这种 BFS 方法保证了只旋转键一侧的原子，不影响另一侧。
    """
    new_coords = coords.copy()
    for (u, v), angle in zip(rot_bonds, delta_angles):
        if abs(angle) < 1e-8:
            continue
        # BFS 从 v 出发，不经过 u，找到 v 侧的所有原子
        visited = set()
        queue = deque([v])
        visited.add(v)
        while queue:
            node = queue.popleft()
            for neighbor in [n.GetIdx() for n in mol.GetAtomWithIdx(node).GetNeighbors()]:
                if neighbor != u and neighbor not in visited:
                    visited.add(neighbor)
                    queue.append(neighbor)
        # 旋转轴: u -> v 方向
        axis = new_coords[v] - new_coords[u]
        axis_norm = np.linalg.norm(axis)
        if axis_norm < 1e-8:
            continue
        axis = axis / axis_norm
        # 用 Rodrigues 公式旋转 v 侧原子（以 u 为支点）
        rot_mat = axis_angle_to_matrix(axis * angle)
        for idx in visited:
            new_coords[idx] = rot_mat @ (new_coords[idx] - new_coords[u]) + new_coords[u]
    return new_coords


def add_noise(coords, tr_sigma, rot_sigma, tor_sigma, mol=None, rot_bonds=None):
    """
    DiffDock 的核心: 三通道独立加噪

    三个通道的物理含义:
      1. 平移 (Translation, R3): 配体质心的位移
         噪声: 质心 += N(0, sigma_tr^2 * I)
      2. 旋转 (Rotation, SO(3)): 配体绕质心的刚体旋转
         噪声: 施加随机轴角旋转，幅度 ~ N(0, sigma_rot^2)
      3. 扭转 (Torsion, S1): 配体内部可旋转键的二面角
         噪声: 每个键的角度 += N(0, sigma_tor^2)，包裹到 [-pi, pi]

    三个通道独立加噪的好处:
      - 平移和旋转是刚体变换，不改变分子内部构象
      - 扭转改变分子柔性部分的构象
      - 独立加噪允许模型分别学习每个自由度的去噪方向
    """
    noised = coords.copy()

    # 1) 平移噪声 -- 整体平移
    tr_noise = np.random.randn(3).astype(np.float32) * tr_sigma
    noised = noised + tr_noise

    # 2) 旋转噪声 -- 绕质心的随机轴角旋转
    rot_noise = np.random.randn(3).astype(np.float32) * rot_sigma
    rot_mat = axis_angle_to_matrix(rot_noise)
    centroid = noised.mean(axis=0)
    noised = (noised - centroid) @ rot_mat.T + centroid

    # 3) 扭转噪声 -- 修改可旋转键角度
    tor_noise = np.array([], dtype=np.float32)
    if mol is not None and rot_bonds is not None and len(rot_bonds) > 0:
        tor_noise = (np.random.randn(len(rot_bonds)) * tor_sigma).astype(np.float32)
        # 包裹到 [-pi, pi]（扭转角的周期性）
        tor_noise = ((tor_noise + np.pi) % (2 * np.pi) - np.pi).astype(np.float32)
        noised = modify_torsion_angles(noised, mol, rot_bonds, tor_noise)

    return noised, tr_noise, rot_noise, tor_noise


# 展示噪声调度
t_vals = np.linspace(0, 1, 50)
schedule_data = []
for t_val in t_vals:
    tr_s, rot_s, tor_s = t_to_sigma(t_val)
    schedule_data.append({'t': t_val, 'tr_sigma': tr_s, 'rot_sigma': rot_s, 'tor_sigma': tor_s})

schedule_df = pd.DataFrame(schedule_data)
print("噪声调度示例（前 5 行 + 后 5 行）:")
display(pd.concat([schedule_df.head(), schedule_df.tail()]))

## 3. 数据集与数据加载器

### 与 Scoring 任务的关键区别

DiffDock 的数据集与 scoring 任务（如 IGN、PIGNet）有几个重要区别：

1. **保留 RDKit 分子对象**：需要用于扭转角操作（BFS 遍历分子图）
2. **保留可旋转键列表**：用于扭转通道的扩散
3. **配体坐标是 ground truth**：训练时通过 `add_noise` 动态生成加噪输入，而非使用固定输入
4. **不预计算交互图边**：模型内部按距离动态构建（因为配体坐标在扩散过程中不断变化）

### 数据流

```
PDB/MOL2 文件 -> build_complex_data() -> DiffDockDataset -> DataLoader
                    |                                          |
            蛋白坐标+特征                                训练时动态加噪
            配体坐标+特征(GT)
            RDKit 分子对象
            可旋转键列表
```

In [ ]:
# ============================
# 数据集构建
# ============================


def build_complex_data(pdbid):
    """
    构建单个蛋白-配体复合物的数据

    与 scoring 任务不同，docking 需要额外保留:
      - 配体的 RDKit 分子对象（用于扭转角操作）
      - 可旋转键列表（用于扭转通道的扩散）
      - 蛋白和配体的 3D 坐标（作为 ground truth 参考）
    """
    pocket_path = os.path.join(COMPLEX_DIR, pdbid, f"{pdbid}_pocket.pdb")
    ligand_mol2 = os.path.join(COMPLEX_DIR, pdbid, f"{pdbid}_ligand.mol2")
    ligand_sdf = os.path.join(COMPLEX_DIR, pdbid, f"{pdbid}_ligand.sdf")

    # ---- 1. 加载蛋白口袋 ----
    prot_mol = Chem.MolFromPDBFile(pocket_path, removeHs=True, sanitize=False)
    if prot_mol is None:
        return None
    try:
        Chem.SanitizeMol(prot_mol)
    except Exception:
        pass

    # ---- 2. 加载配体 (mol2 优先, sdf 备选) ----
    lig_mol = Chem.MolFromMol2File(ligand_mol2, sanitize=False)
    if lig_mol is None and os.path.exists(ligand_sdf):
        lig_mol = load_mol(ligand_sdf, 'sdf')
    if lig_mol is None:
        return None
    try:
        Chem.SanitizeMol(lig_mol)
    except Exception:
        pass

    # ---- 3. 去氢 ----
    try:
        prot_mol = Chem.RemoveHs(prot_mol)
    except Exception:
        pass
    try:
        lig_mol = Chem.RemoveHs(lig_mol)
    except Exception:
        pass

    # ---- 4. 提取 3D 坐标与原子特征 ----
    prot_conf = prot_mol.GetConformer()
    lig_conf = lig_mol.GetConformer()

    prot_coords = np.array([prot_conf.GetAtomPosition(i)
                            for i in range(prot_mol.GetNumAtoms())], dtype=np.float32)
    lig_coords = np.array([lig_conf.GetAtomPosition(i)
                           for i in range(lig_mol.GetNumAtoms())], dtype=np.float32)

    prot_feats = np.array([atom_features(a) for a in prot_mol.GetAtoms()], dtype=np.float32)
    lig_feats = np.array([atom_features(a) for a in lig_mol.GetAtoms()], dtype=np.float32)

    if prot_feats.shape[0] == 0 or lig_feats.shape[0] == 0:
        return None

    # ---- 5. 获取可旋转键 ----
    rot_bonds = get_rotatable_bonds(lig_mol)

    return {
        'prot_coords': prot_coords,     # (Np, 3)
        'lig_coords': lig_coords,        # (Nl, 3) -- ground truth 坐标
        'prot_feats': prot_feats,        # (Np, 10)
        'lig_feats': lig_feats,          # (Nl, 10)
        'mol': lig_mol,                  # RDKit 分子对象（用于扭转操作）
        'rot_bonds': rot_bonds,          # 可旋转键列表
    }


class DiffDockDataset(Dataset):
    """
    扩散对接数据集

    与 scoring 数据集的关键区别:
      - 保留配体的 RDKit 分子对象和可旋转键信息
      - 配体坐标是 ground truth（训练时通过加噪动态生成输入）
      - 不需要预计算交互图边（模型内部按距离动态构建）
    """
    def __init__(self, data_list):
        # data_list: [(complex_data_dict, logKa), ...]
        self.data = data_list

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        cdata, logKa = self.data[idx]
        return {
            'prot_coords': torch.FloatTensor(cdata['prot_coords']),
            'lig_coords': torch.FloatTensor(cdata['lig_coords']),
            'prot_feats': torch.FloatTensor(cdata['prot_feats']),
            'lig_feats': torch.FloatTensor(cdata['lig_feats']),
            'mol': cdata['mol'],
            'rot_bonds': cdata['rot_bonds'],
            'label': torch.FloatTensor([logKa]),
        }


def collate_fn(batch):
    """BATCH_SIZE=1 时直接返回单个样本（变长数据无法堆叠）"""
    return batch[0]


# ---- 预处理所有复合物 ----
print("正在构建扩散对接数据...")
labels = parse_coreset(CORESET_FILE)

all_data = []
failed = 0
for pdbid, logKa in sorted(labels.items()):
    result = build_complex_data(pdbid)
    if result is None:
        failed += 1
        continue
    all_data.append((result, logKa))

print(f"成功加载 {len(all_data)} 个复合物, 失败 {failed} 个")

# ---- 随机 80/20 划分训练/测试集 ----
indices = np.random.permutation(len(all_data))
split = int(0.8 * len(all_data))
train_idx, test_idx = indices[:split], indices[split:]

train_data = [all_data[i] for i in train_idx]
test_data = [all_data[i] for i in test_idx]

print(f"训练集: {len(train_data)}, 测试集: {len(test_data)}")

train_loader = DataLoader(DiffDockDataset(train_data), batch_size=1,
                          shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(DiffDockDataset(test_data), batch_size=1,
                         shuffle=False, collate_fn=collate_fn)

# 展示一个样本的数据形状
sample = next(iter(train_loader))
shape_info = pd.DataFrame({
    '数据项': ['蛋白坐标', '蛋白特征', '配体坐标(GT)', '配体特征', '可旋转键数'],
    '形状/值': [
        str(tuple(sample['prot_coords'].shape)),
        str(tuple(sample['prot_feats'].shape)),
        str(tuple(sample['lig_coords'].shape)),
        str(tuple(sample['lig_feats'].shape)),
        str(len(sample['rot_bonds'])),
    ]
})
print("\n样本数据形状:")
display(shape_info)

## 4. 模型架构

### 整体架构

ToyDiffDockScore 是 DiffDock 去噪分数模型的简化版本，核心思想是：

> 给定加噪后的蛋白-配体复合物和噪声水平 sigma(t)，模型预测三个方向的去噪「分数」
> (score = nabla_x log p)

三个分数的含义：
- **`tr_score` (3,)**：平移分数 -- 指向正确质心的方向
- **`rot_score` (3,)**：旋转分数（轴角表示）-- 指向正确朝向
- **`tor_score` (n_tor,)**：扭转分数 -- 指向正确二面角

### 逐层分解

```
蛋白原子特征 (Np, 10)  --> prot_embed --> (Np, 128) --+
                                                       |
配体原子特征 (Nl, 10)  --> lig_embed  --> (Nl, 128) --+
                                                       |  + time_embed(t)
时间步 t in [0,1]      --> SinusoidalEmbedding --> (128) --+
                                                       |
                                              4 x InteractionLayer
                                              (距离感知消息传递)
                                                       |
                              +-------------+----------+----------+
                              |             |                     |
                          tr_head       rot_head              tor_head
                        (全局池化->3D)  (全局池化->3D)        (逐原子->1D)
                              |             |                     |
                        tr_score(3,)  rot_score(3,)       tor_score(n_tor,)
```

### InteractionLayer: 距离感知的消息传递

与 IGN 的关键区别：IGN 在预处理时构建固定的交互图，而 DiffDock 在每一层**动态计算**交互边（因为配体坐标在扩散过程中不断变化）。

每一层的计算流程：
1. 计算蛋白原子与配体原子间的距离矩阵
2. 选择距离 < `DISTANCE_CUTOFF` 的原子对作为交互边
3. 用距离加权的消息传递更新双方节点特征
4. 残差连接 + LayerNorm

In [ ]:
# ============================
# InteractionLayer: 蛋白-配体交互层
# ============================


class InteractionLayer(nn.Module):
    """
    蛋白-配体交互层（简化版）

    核心机制（模仿 DiffDock 的图注意力交互）:
      1. 计算蛋白原子与配体原子间的距离矩阵
      2. 选择距离 < DISTANCE_CUTOFF 的原子对作为交互边
      3. 用距离加权的消息传递更新双方节点特征

    与 IGN 的区别:
      - IGN 在预处理时构建固定的交互图
      - DiffDock 在每一层动态计算（因为配体坐标在扩散过程中不断变化）
    """
    def __init__(self, hidden_dim):
        super().__init__()
        # 消息 MLP: [源节点特征 || 目标节点特征 || 距离] -> 消息
        self.msg_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2 + 1, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        # 更新 MLP: [原特征 || 聚合消息] -> 更新后特征
        self.update_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.layer_norm = nn.LayerNorm(hidden_dim)

    def _scatter_aggregate(self, src_h, dst_h, dst_idx, distances, n_dst):
        """
        向量化的消息传递与聚合（避免 Python for 循环）

        利用 scatter_add_ 在 C 层面完成按目标节点的加权聚合，
        比逐节点 Python 循环快 10-100 倍。
        """
        hidden = src_h.shape[1]
        d = distances.unsqueeze(-1)                                     # (E, 1)
        msg = self.msg_mlp(torch.cat([src_h, dst_h, d], dim=-1))       # (E, hidden)
        w = 1.0 / (d + 1e-3)                                           # (E, 1)
        # 按目标节点归一化权重
        w_sum = torch.zeros(n_dst, 1, device=src_h.device)
        idx_expand_1 = dst_idx.unsqueeze(-1).expand(-1, 1)
        w_sum.scatter_add_(0, idx_expand_1, w)
        w_norm = w / (w_sum[dst_idx] + 1e-8)                           # (E, 1)
        # scatter 加权消息
        agg = torch.zeros(n_dst, hidden, device=src_h.device)
        idx_expand_h = dst_idx.unsqueeze(-1).expand(-1, hidden)
        agg.scatter_add_(0, idx_expand_h, w_norm * msg)
        return agg

    def forward(self, prot_h, lig_h, prot_pos, lig_pos):
        """
        参数:
          prot_h:   (Np, hidden) 蛋白节点特征
          lig_h:    (Nl, hidden) 配体节点特征
          prot_pos: (Np, 3) 蛋白原子坐标
          lig_pos:  (Nl, 3) 配体原子坐标（加噪后，动态变化）
        返回:
          更新后的 (prot_h, lig_h)
        """
        # 计算蛋白-配体距离矩阵
        dist = torch.cdist(lig_pos.unsqueeze(0), prot_pos.unsqueeze(0)).squeeze(0)  # (Nl, Np)
        mask = dist < DISTANCE_CUTOFF  # 交互掩码

        # 找出所有交互边 (向量化，替代逐原子 for 循环)
        lig_idx, prot_idx = torch.where(mask)               # (E,) 每条边的配体/蛋白原子索引
        edge_dist = dist[lig_idx, prot_idx]                  # (E,) 边距离

        # --- 更新配体节点: 从附近蛋白原子聚合信息 ---
        if lig_idx.shape[0] > 0:
            lig_agg = self._scatter_aggregate(
                prot_h[prot_idx], lig_h[lig_idx], lig_idx, edge_dist, lig_h.shape[0])
        else:
            lig_agg = torch.zeros_like(lig_h)

        # --- 更新蛋白节点: 从附近配体原子聚合信息 ---
        if lig_idx.shape[0] > 0:
            prot_agg = self._scatter_aggregate(
                lig_h[lig_idx], prot_h[prot_idx], prot_idx, edge_dist, prot_h.shape[0])
        else:
            prot_agg = torch.zeros_like(prot_h)

        # 残差连接 + LayerNorm
        lig_h = self.layer_norm(lig_h + self.update_mlp(torch.cat([lig_h, lig_agg], dim=-1)))
        prot_h = self.layer_norm(prot_h + self.update_mlp(torch.cat([prot_h, prot_agg], dim=-1)))
        return prot_h, lig_h

In [ ]:
# ============================
# ToyDiffDockScore: 去噪分数模型
# ============================


class ToyDiffDockScore(nn.Module):
    """
    DiffDock 去噪分数模型（简化版）

    核心思想:
      给定加噪后的蛋白-配体复合物和噪声水平 sigma(t)，
      模型预测三个方向的去噪「分数」(score = nabla_x log p):
        - tr_score:  (3,)     平移分数 -- 指向正确质心的方向
        - rot_score: (3,)     旋转分数（轴角表示）-- 指向正确朝向
        - tor_score: (n_tor,) 扭转分数 -- 指向正确二面角

    架构:
      1. 节点嵌入: 蛋白/配体原子特征 -> 隐空间（独立嵌入层）
      2. 时间嵌入: 正弦编码 t，注入到节点特征中（告知模型当前噪声水平）
      3. 交互层: 4 层距离感知的蛋白-配体消息传递
      4. 三个预测头: 分别预测 tr/rot/tor 分数

    分数的物理含义:
      score = -noise / sigma^2，即噪声的负方向除以方差
      模型学习的是「去掉噪声的方向和大小」
    """
    def __init__(self, atom_dim=ATOM_FEAT_DIM, hidden_dim=HIDDEN_DIM, n_layers=4):
        super().__init__()
        # 时间步嵌入 -- 正弦编码
        self.time_embed = SinusoidalEmbedding(hidden_dim)

        # 节点嵌入（蛋白和配体使用不同的嵌入层）
        self.prot_embed = nn.Linear(atom_dim, hidden_dim)
        self.lig_embed = nn.Linear(atom_dim, hidden_dim)

        # 时间注入 MLP -- 将时间信息融合到节点特征
        self.time_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        # 4 层蛋白-配体交互
        self.interaction_layers = nn.ModuleList([
            InteractionLayer(hidden_dim) for _ in range(n_layers)
        ])

        # 三个独立的预测头
        self.tr_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 3),       # 平移分数 (3,)
        )
        self.rot_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 3),       # 旋转分数 (3,) -- 轴角表示
        )
        self.tor_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1),       # 每个原子 -> 1 维扭转分数
        )

    def forward(self, prot_feats, lig_feats, prot_pos, lig_pos, t, n_tor=0):
        """
        参数:
          prot_feats: (Np, atom_dim) 蛋白原子特征
          lig_feats:  (Nl, atom_dim) 配体原子特征
          prot_pos:   (Np, 3)       蛋白原子坐标
          lig_pos:    (Nl, 3)       配体原子坐标（加噪后）
          t:          float/Tensor  当前时间步, t in [0, 1]
          n_tor:      int           可旋转键数量
        返回:
          tr_score:  (3,)     平移分数
          rot_score: (3,)     旋转分数
          tor_score: (n_tor,) 扭转分数
        """
        # Step 1: 节点嵌入
        prot_h = self.prot_embed(prot_feats)        # (Np, hidden)
        lig_h = self.lig_embed(lig_feats)           # (Nl, hidden)

        # Step 2: 时间嵌入注入 -- 让模型感知当前噪声水平
        if not isinstance(t, torch.Tensor):
            t_tensor = torch.tensor([t], dtype=torch.float32, device=prot_feats.device)
        else:
            t_tensor = t.view(-1).to(prot_feats.device)
        t_emb = self.time_mlp(self.time_embed(t_tensor))  # (1, hidden)
        lig_h = lig_h + t_emb          # 广播: (Nl, hidden) + (1, hidden)
        prot_h = prot_h + t_emb

        # Step 3: 4 层蛋白-配体交互
        for layer in self.interaction_layers:
            prot_h, lig_h = layer(prot_h, lig_h, prot_pos, lig_pos)

        # Step 4: 三个预测头
        # 平移分数: 配体全局特征（均值池化）-> 3 维
        lig_global = lig_h.mean(dim=0, keepdim=True)
        tr_score = self.tr_head(lig_global).squeeze(0)      # (3,)

        # 旋转分数: 配体全局特征 -> 3 维轴角向量
        rot_score = self.rot_head(lig_global).squeeze(0)     # (3,)

        # 扭转分数: 每个配体原子 -> 1 维，取前 n_tor 个作为各键的分数
        # （简化处理: 实际 DiffDock 按键端原子聚合，这里取前 n_tor 个原子输出）
        if n_tor > 0:
            tor_per_atom = self.tor_head(lig_h).squeeze(-1)  # (Nl,)
            if tor_per_atom.shape[0] >= n_tor:
                tor_score = tor_per_atom[:n_tor]
            else:
                tor_score = torch.zeros(n_tor, device=prot_feats.device)
                tor_score[:tor_per_atom.shape[0]] = tor_per_atom
        else:
            tor_score = torch.zeros(0, device=prot_feats.device)

        return tr_score, rot_score, tor_score


# ---- 模型实例化 ----
model = ToyDiffDockScore(atom_dim=ATOM_FEAT_DIM, hidden_dim=HIDDEN_DIM).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# 参数量统计
param_counts = []
for name, param in model.named_parameters():
    param_counts.append({
        '模块': name.split('.')[0],
        '参数名': name,
        '形状': str(tuple(param.shape)),
        '参数量': param.numel()
    })
param_df = pd.DataFrame(param_counts)
total_params = sum(p.numel() for p in model.parameters())

# 按模块汇总
module_summary = param_df.groupby('模块')['参数量'].sum().reset_index()
module_summary.columns = ['模块', '参数量']
module_summary = module_summary.sort_values('参数量', ascending=False)

print(f"模型总参数量: {total_params:,}")
print("\n各模块参数量:")
display(module_summary)

## 5. 训练

### 去噪分数匹配损失 (Denoising Score Matching)

这是扩散模型训练的核心损失函数。

#### 理论推导

给定干净数据 x0，前向扩散添加噪声 epsilon：

    x_noisy = x0 + epsilon,  epsilon ~ N(0, sigma^2 * I)

对数似然的梯度（即「分数」）为：

    nabla_x log p(x_noisy | x0) = -(x_noisy - x0) / sigma^2 = -epsilon / sigma^2

因此，去噪分数匹配的目标是：

    L = ||score_pred - score_true||^2

其中 score_true = -epsilon / sigma^2

#### 三通道损失

对三个通道分别计算：
- 平移: score_tr = -epsilon_tr / sigma_tr^2
- 旋转: score_rot = -epsilon_rot / sigma_rot^2
- 扭转: score_tor = -epsilon_tor / sigma_tor^2

总损失 = L_tr + L_rot + L_tor

#### 训练技巧

- **梯度裁剪** (`clip_grad_norm_`)：扩散模型训练中常用的稳定化技巧，防止某些噪声水平下梯度爆炸
- **较小学习率** (1e-4)：扩散模型比判别模型更需要稳定的更新

In [ ]:
# ============================
# 去噪分数匹配损失 + 训练循环
# ============================


def score_matching_loss(model, data, device):
    """
    去噪分数匹配损失 (Denoising Score Matching)

    对三个通道分别计算:
      - 平移: score_tr = -tr_noise / sigma_tr^2
      - 旋转: score_rot = -rot_noise / sigma_rot^2
      - 扭转: score_tor = -tor_noise / sigma_tor^2
    """
    prot_feats = data['prot_feats'].to(device)
    lig_feats = data['lig_feats'].to(device)
    prot_pos = data['prot_coords'].to(device)
    lig_pos_clean = data['lig_coords']
    mol = data['mol']
    rot_bonds = data['rot_bonds']
    n_tor = len(rot_bonds)

    # 1. 随机采样时间步 t ~ U(0, 1)
    t = np.random.uniform(0.0, 1.0)
    tr_sigma, rot_sigma, tor_sigma = t_to_sigma(t)

    # 2. 对配体坐标施加三通道噪声（在 numpy 空间操作）
    lig_coords_np = lig_pos_clean.cpu().numpy() if isinstance(lig_pos_clean, torch.Tensor) else lig_pos_clean
    noised_coords, tr_noise, rot_noise, tor_noise = add_noise(
        lig_coords_np.copy(), tr_sigma, rot_sigma, tor_sigma, mol, rot_bonds
    )
    noised_pos = torch.FloatTensor(noised_coords).to(device)

    # 3. 计算真实分数 (ground truth)
    #    score_true = -noise / sigma^2
    tr_score_true = torch.FloatTensor(-tr_noise / (tr_sigma ** 2 + 1e-8)).to(device)
    rot_score_true = torch.FloatTensor(-rot_noise / (rot_sigma ** 2 + 1e-8)).to(device)

    # 4. 模型预测分数
    tr_pred, rot_pred, tor_pred = model(
        prot_feats, lig_feats, prot_pos, noised_pos, t, n_tor
    )

    # 5. 三通道 MSE 损失
    loss_tr = ((tr_pred - tr_score_true) ** 2).mean()
    loss_rot = ((rot_pred - rot_score_true) ** 2).mean()

    loss_tor = torch.tensor(0.0, device=device)
    if n_tor > 0 and len(tor_noise) > 0:
        tor_score_true = torch.FloatTensor(-tor_noise / (tor_sigma ** 2 + 1e-8)).to(device)
        loss_tor = ((tor_pred - tor_score_true) ** 2).mean()

    # 总损失 = 三通道之和
    return loss_tr + loss_rot + loss_tor


# ---- 训练循环 ----
print(f"模型参数量: {sum(p.numel() for p in model.parameters()):,}")
print(f"开始训练 {N_EPOCHS} 轮...\n")

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    train_losses = []

    for data in train_loader:
        optimizer.zero_grad()
        loss = score_matching_loss(model, data, DEVICE)
        loss.backward()
        # 梯度裁剪 -- 扩散模型训练中常用的稳定化技巧
        # 防止某些噪声水平下梯度爆炸
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_losses.append(loss.item())

    # ---- 每 20 轮评估一次 ----
    if epoch % 20 == 0 or epoch == 1:
        model.eval()
        val_losses = []
        with torch.no_grad():
            for data in test_loader:
                vloss = score_matching_loss(model, data, DEVICE)
                val_losses.append(vloss.item())

        avg_train = np.mean(train_losses) if train_losses else float('nan')
        avg_val = np.mean(val_losses) if val_losses else float('nan')
        print(f"  Epoch {epoch:>4d}/{N_EPOCHS}  |  Train Loss: {avg_train:.4f}  |  Val Loss: {avg_val:.4f}")

## 6. 评估与可视化

### 逆扩散推理 (Reverse Diffusion Inference)

这是 DiffDock 推理的核心算法，模拟从随机位姿迭代恢复正确结合构象的过程：

1. **初始化**：对 ground truth 坐标施加最大噪声，生成随机位姿
2. **迭代去噪**：从 t=1 逐步降低到 t 接近 0，每一步：
   - 计算当前噪声水平 sigma(t)
   - 用模型预测去噪分数
   - 沿分数方向更新位姿（简化 Langevin 动力学）
3. **返回**最终去噪后的配体坐标

更新规则（简化的 Langevin 步骤）：

    delta_x = sigma^2 * dt * score_pred

其中 dt = 1/n_steps 是时间步长。

### 评估指标

- **RMSD (Root Mean Square Deviation)**：预测位姿与 ground truth 之间的均方根偏差
  - RMSD < 2 A：通常认为是成功对接
  - RMSD < 5 A：可接受的对接结果
- **去噪轨迹**：单个样本的 RMSD 随推理步骤的变化，展示模型逐步恢复正确构象的过程

In [ ]:
# ============================
# 逆扩散推理
# ============================


def inference(model, data, device, n_steps=N_STEPS):
    """
    逆扩散推理 -- 从随机位姿迭代去噪恢复结合构象

    更新规则 (简化的 Langevin 步骤):
      delta_x = sigma^2 * dt * score_pred
    其中 dt = 1/n_steps 是时间步长。

    返回: (最终坐标, RMSD 轨迹)
    """
    model.eval()
    prot_feats = data['prot_feats'].to(device)
    lig_feats = data['lig_feats'].to(device)
    prot_pos = data['prot_coords'].to(device)
    lig_pos_clean = data['lig_coords'].cpu().numpy() if isinstance(
        data['lig_coords'], torch.Tensor) else data['lig_coords']
    mol = data['mol']
    rot_bonds = data['rot_bonds']
    n_tor = len(rot_bonds)

    # 1. 初始化: 施加最大噪声生成随机位姿
    noised_coords, _, _, _ = add_noise(
        lig_pos_clean.copy(), TR_SIGMA_MAX, ROT_SIGMA_MAX, TOR_SIGMA_MAX,
        mol, rot_bonds
    )

    # 时间步调度: 从 1 线性降低到接近 0
    t_schedule = np.linspace(1.0, 0.01, n_steps)
    dt = 1.0 / n_steps
    rmsd_traj = []

    with torch.no_grad():
        for t in t_schedule:
            tr_sigma, rot_sigma, tor_sigma = t_to_sigma(t)
            lig_pos_noisy = torch.FloatTensor(noised_coords).to(device)

            # 模型预测去噪分数
            tr_score, rot_score, tor_score = model(
                prot_feats, lig_feats, prot_pos, lig_pos_noisy, t, n_tor
            )

            # 平移更新: delta_x = sigma_tr^2 * dt * tr_score
            tr_update = (tr_sigma ** 2) * dt * tr_score.cpu().numpy()
            noised_coords = noised_coords + tr_update

            # 旋转更新: 将 sigma_rot^2 * dt * rot_score 作为轴角旋转施加到配体
            rot_update = (rot_sigma ** 2) * dt * rot_score.cpu().numpy()
            rot_mat = axis_angle_to_matrix(rot_update)
            centroid = noised_coords.mean(axis=0)
            noised_coords = (noised_coords - centroid) @ rot_mat.T + centroid

            # 扭转更新: 修改可旋转键角度
            if n_tor > 0 and tor_score.shape[0] > 0:
                tor_update = (tor_sigma ** 2) * dt * tor_score.cpu().numpy()
                noised_coords = modify_torsion_angles(
                    noised_coords, mol, rot_bonds, tor_update
                )

            # 记录当前 RMSD（与 ground truth 的均方根偏差）
            rmsd = np.sqrt(np.mean(np.sum(
                (noised_coords - lig_pos_clean) ** 2, axis=1)))
            rmsd_traj.append(rmsd)

    return noised_coords, rmsd_traj

In [ ]:
# ============================
# 在测试集上推理并可视化结果
# ============================

print("在测试集上进行逆扩散推理...")

model.eval()
all_rmsds = []
sample_traj = None          # 保存一个样本的去噪轨迹用于绘图

for i, data in enumerate(test_loader):
    final_coords, rmsd_traj = inference(model, data, DEVICE, n_steps=N_STEPS)
    final_rmsd = rmsd_traj[-1]
    all_rmsds.append(final_rmsd)
    if sample_traj is None:
        sample_traj = rmsd_traj

all_rmsds = np.array(all_rmsds)
median_rmsd = np.median(all_rmsds)
mean_rmsd = np.mean(all_rmsds)
pct_below_2 = np.mean(all_rmsds < 2.0) * 100
pct_below_5 = np.mean(all_rmsds < 5.0) * 100

# 用 DataFrame 展示结果
results_df = pd.DataFrame({
    '指标': ['测试样本数', 'Median RMSD', 'Mean RMSD', 'RMSD < 2A', 'RMSD < 5A'],
    '值': [
        f'{len(all_rmsds)}',
        f'{median_rmsd:.2f} A',
        f'{mean_rmsd:.2f} A',
        f'{pct_below_2:.1f}%',
        f'{pct_below_5:.1f}%',
    ]
})
print("\n逆扩散推理结果 (Reverse Diffusion Results):")
display(results_df)

# ---- 可视化: 2 个子图 ----
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 左图: RMSD 分布直方图
ax1 = axes[0]
ax1.hist(all_rmsds, bins=20, color='steelblue', edgecolor='black', alpha=0.7)
ax1.axvline(x=2.0, color='red', linestyle='--', linewidth=1.5, label='2 A')
ax1.axvline(x=5.0, color='orange', linestyle='--', linewidth=1.5, label='5 A')
ax1.set_xlabel('RMSD (A)', fontsize=12)
ax1.set_ylabel('Count', fontsize=12)
ax1.set_title(f'DiffDock Toy: RMSD Distribution\n'
              f'Median={median_rmsd:.2f}A, <2A: {pct_below_2:.1f}%', fontsize=11)
ax1.legend()

# 右图: 去噪轨迹 (单个样本的 RMSD 随推理步骤的变化)
ax2 = axes[1]
if sample_traj is not None:
    steps = np.arange(1, len(sample_traj) + 1)
    ax2.plot(steps, sample_traj, 'b-o', markersize=3, linewidth=1.5)
    ax2.set_xlabel('Denoising Step', fontsize=12)
    ax2.set_ylabel('RMSD (A)', fontsize=12)
    ax2.set_title('Denoising Trajectory (Sample)', fontsize=11)
    ax2.axhline(y=2.0, color='red', linestyle='--', linewidth=1.0,
                alpha=0.5, label='2 A threshold')
    ax2.legend()

plt.tight_layout()
plt.show()

## 总结

### 本教程的关键要点

1. **SE(3) 扩散的三通道设计**：DiffDock 将配体位姿的三个自由度（平移、旋转、扭转）分别建模，每个通道有独立的噪声调度。这种设计尊重了不同自由度的物理特性和几何结构。

2. **去噪分数匹配**：模型学习的是 score = -epsilon / sigma^2，即从噪声数据恢复干净数据的方向。三个通道分别预测平移分数、旋转分数和扭转分数。

3. **动态交互图**：与 IGN 等 scoring 模型的固定交互图不同，DiffDock 在每一层都根据当前（加噪后的）配体坐标重新计算蛋白-配体交互边，因为配体位置在扩散过程中不断变化。

4. **逆扩散推理**：从随机位姿出发，通过迭代去噪逐步恢复正确的结合构象。每一步使用简化的 Langevin 动力学更新：delta_x = sigma^2 * dt * score。

### 教学版 vs. 完整版的主要简化

| 方面 | 教学版 (本教程) | 完整版 (DiffDock) |
|------|----------------|-------------------|
| 原子特征 | 10 维 (元素 one-hot + 芳香性) | ESM 蛋白嵌入 + RDKit 化学特征 |
| 交互层 | 4 层距离感知 MLP | SE(3)-等变的 Tensor Field Networks |
| 扭转预测 | 取前 n_tor 个原子输出 | 按键端原子聚合 |
| 推理策略 | 简化 Langevin 步骤 | 完整的 ODE/SDE 求解器 |
| 置信度 | 无 | 额外的置信度模型排序候选位姿 |
| 数据集 | CASF-2016 core set (285) | PDBBind + 数据增强 |